# 10 -- Blending и stacking

Проверяю, дополняют ли модели друг друга. Простое смешивание рангов -- первая альтернатива.
Stacking требует отдельной выборки для метамодели: на обучающих предсказаниях базовых моделей его не обучаю.
Делю dev пополам по запросам: calibration и comparison. Оба поднабора уже участвовали в прежних
экспериментах выбора базовых моделей, поэтому comparison -- диагностика, не новый независимый holdout.

In [1]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from IPython.display import display

en_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
get_ipython().run_line_magic('run', str(en_root / 'notebooks/common.ipynb'))
en_out = en_root / 'outputs'
en_seed = 143
en_scores = pd.read_parquet(en_out / 'cache/extended_dev_scores.parquet')
en_queries = pd.read_parquet(en_out / 'cache/eval_queries.parquet').query(
    "split == 'dev'"
)
en_items = pd.read_parquet(en_out / 'cache/items.parquet', columns=['item_id'])
en_truth_all = json.loads((en_out / 'cache/truth.json').read_text())
en_truth = {q: en_truth_all[q] for q in en_queries.query_id}
en_warm_share = json.loads((en_out / 'models/hybrid.json').read_text())['warm_share']


## 1. Разделение для метамодели

In [2]:
en_cal_ids = (
    en_queries.groupby('regime', group_keys=False)
    .sample(frac=0.5, random_state=en_seed)
    .query_id
)


In [3]:
en_queries = en_queries.assign(
    meta_part=np.where(
        en_queries.query_id.isin(en_cal_ids), 'calibration', 'comparison'
    )
)
assert not set(en_cal_ids) & set(
    en_queries.loc[en_queries.meta_part.eq('comparison'), 'query_id']
)
en_queries[['query_id', 'regime', 'meta_part']].to_csv(
    en_out / 'validation/meta_split.csv', index=False
)
en_models = ['original_300', 'extended_300', 'ranker_300', 'mlp_6']
en_rank_features = pd.DataFrame(index=en_scores.index)
for en_name in en_models:
    en_ordered = en_scores.sort_values(
        ['query_id', en_name, 'pos'], ascending=[True, False, True]
    )
    en_rank = en_ordered.groupby('query_id').cumcount() + 1
    en_rank_features[en_name] = (31 / (30 + en_rank)).reindex(en_scores.index)
en_results = []


In [4]:
def en_evaluate(name, values):
    predictions = predictions_from_frame(
        en_scores.assign(score=values), en_items, score='score', k=100
    )
    metrics = recall_table(predictions, en_truth).merge(
        en_queries[['query_id', 'regime', 'meta_part']], on='query_id'
    )
    for part, group in metrics.groupby('meta_part'):
        scores = group.groupby('regime')['recall@50'].mean()
        en_results.append(
            {
                'variant': name,
                'part': part,
                'cold': scores['cold'],
                'warm': scores['warm'],
                'objective': (1 - en_warm_share) * scores['cold']
                + en_warm_share * scores['warm'],
            }
        )
    return metrics


## 2. Смешивание рангов

Проверяю небольшую сетку весов. Выбираю вес только по calibration, затем показываю comparison.
Скоры моделей не складываю напрямую: у Logloss, QuerySoftMax и MLP разные шкалы.

In [5]:
en_values = {}
for en_model in ['extended_300', 'ranker_300', 'mlp_6']:
    for en_alpha in [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]:
        en_name = f'{en_model}_{en_alpha:g}'
        en_values[en_name] = (
            1 - en_alpha
        ) * en_rank_features.original_300 + en_alpha * en_rank_features[en_model]
        en_evaluate(en_name, en_values[en_name])
en_blends = pd.DataFrame(en_results)
en_selected = (
    en_blends.query("part == 'calibration'")
    .sort_values(['objective', 'variant'], ascending=[False, True])
    .iloc[0]
    .variant
)
display(en_blends[en_blends.variant.eq(en_selected)])


,variant,part,cold,warm,objective
8,extended_300_0.75,calibration,0.811667,0.828857,0.818110
9,extended_300_0.75,comparison,0.785944,0.801000,0.791587


## 3. Логистическая метамодель

Вход -- четыре ранговых признака. Базовые модели не обучались на dev, поэтому метамодель
получает отложенные предсказания. Calibration использую для fit, comparison -- только для диагностики.
Сравниваю с простым blending: дополнительный уровень должен оправдать себя.

In [6]:
en_target = np.array(
    [
        int(en_items.item_id.iloc[p] in en_truth[q])
        for q, p in zip(en_scores.query_id, en_scores.pos)
    ]
)
en_cal_mask = en_scores.query_id.isin(en_cal_ids).to_numpy()
en_labels = en_scores[['query_id']].assign(target=en_target)
en_npos = en_labels.groupby('query_id').target.transform('sum')
en_n = en_labels.groupby('query_id').target.transform('size')
en_fit_mask = en_cal_mask & en_npos.gt(0).to_numpy()
en_weights = np.where(
    en_target == 1, 0.5 / en_npos.clip(lower=1), 0.5 / (en_n - en_npos).clip(lower=1)
)
en_stack = LogisticRegression(C=1.0, max_iter=300, random_state=en_seed)
en_stack.fit(
    en_rank_features.loc[en_fit_mask],
    en_target[en_fit_mask],
    sample_weight=en_weights[en_fit_mask] / en_weights[en_fit_mask].mean(),
)
en_stack_metrics = en_evaluate('stacking', en_stack.decision_function(en_rank_features))
en_results = pd.DataFrame(en_results)
en_results.to_csv(en_out / 'validation/ensemble_experiments.csv', index=False)
display(
    en_results[en_results.variant.isin([en_selected, 'stacking', 'extended_300_0'])]
)


,variant,part,cold,warm,objective
0,extended_300_0,calibration,0.806667,0.821571,0.812253
1,extended_300_0,comparison,0.794611,0.797500,0.795694
8,extended_300_0.75,calibration,0.811667,0.828857,0.818110
9,extended_300_0.75,comparison,0.785944,0.801000,0.791587
36,stacking,calibration,0.811667,0.820571,0.815004
37,stacking,comparison,0.791278,0.805500,0.796608


## 4. Отдельный кандидат с расширенными признаками

Для воспроизводимого сравнения сохраняю CatBoost с расширенными признаками на 300 деревьях.
После всех локальных оценок обучаю его на доступных отложенных кандидатах трех частей,
как делал для исходной модели. Полученные модели не оцениваю снова на этих же частях.
Stacking в финальный ответ автоматически не переношу: после refit меняется распределение его входов.

In [7]:
en_columns = json.loads((en_out / 'models/extended_features.json').read_text())[
    'features'
]
en_catalog = pd.read_parquet(en_out / 'cache/items.parquet')
en_train = pd.concat(
    [
        pd.read_parquet(en_out / f'cache/extended_{part}.parquet')
        for part in ['rank_train', 'dev', 'test']
    ],
    ignore_index=True,
)
en_ids = en_catalog.item_id.to_numpy()
en_train['target'] = [
    int(en_ids[p] in en_truth_all[q]) for q, p in zip(en_train.query_id, en_train.pos)
]
en_positive = en_train.groupby('query_id').target.transform('sum')
en_train = en_train[en_positive.gt(0)].copy()
en_positive = en_train.groupby('query_id').target.transform('sum')
en_n = en_train.groupby('query_id').target.transform('size')
en_weight = np.where(
    en_train.target.eq(1), 0.5 / en_positive, 0.5 / (en_n - en_positive).clip(lower=1)
)
en_final = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.07,
    loss_function='Logloss',
    l2_leaf_reg=5,
    random_seed=en_seed,
    thread_count=4,
    allow_writing_files=False,
    verbose=100,
)
en_final.fit(
    model_matrix(en_train, en_columns),
    en_train.target,
    sample_weight=en_weight / en_weight.mean(),
)
en_final.save_model(str(en_out / 'models/extended_final.cbm'))
del en_train


0:	learn: 0.6201075	total: 394ms	remaining: 1m 57s


100:	learn: 0.2053228	total: 33.3s	remaining: 1m 5s


200:	learn: 0.1874160	total: 1m 4s	remaining: 31.9s


299:	learn: 0.1710401	total: 1m 35s	remaining: 0us


## 5. Benchmark: те же признаки и проверка исходной выдачи

Перед новым ответом проверяю точное восстановление исходного CSV. Это проверка порядка items,
общих признаков и использования финальных весов. Исходный файл сохраняю без изменений.

In [8]:
en_raw, en_bq_raw, en_bi_raw = load_raw()
en_bq = prepare_queries(en_bq_raw)
en_positions = np.flatnonzero(en_catalog.item_id.isin(en_bi_raw.item_id).to_numpy())
en_bitems = en_catalog.iloc[en_positions].reset_index(drop=True)
en_numeric = (
    pd.read_parquet(en_out / 'cache/item_features.parquet')
    .iloc[en_positions]
    .reset_index(drop=True)
)
en_pairs = prepare_queries(en_raw)
en_pairs['item_id'] = en_raw.item_id.to_numpy()
en_pairs = en_pairs.drop_duplicates(['context_key', 'item_id'])
en_counts = en_pairs.item_id.value_counts().to_dict()
en_embedding_ids = pd.read_parquet(
    en_out / 'cache/embedding_query_ids.parquet'
).query_id.tolist()
en_qvectors = dict(
    zip(en_embedding_ids, np.load(en_out / 'cache/query_embeddings.npy'))
)
en_ivectors = np.asarray(
    np.load(en_out / 'cache/item_embeddings.npy', mmap_mode='r')[en_positions]
)
en_config = json.loads((en_out / 'models/selected.json').read_text())
en_centers = en_catalog.groupby('item_location_id')[
    ['item_latitude', 'item_longitude']
].median()


In [9]:
en_candidates = pd.read_parquet(en_out / 'cache/benchmark_candidates.parquet')
en_frame = candidate_features(
    en_candidates,
    en_bq,
    en_bitems,
    en_numeric,
    en_counts,
    en_config['weights'],
    en_qvectors,
    en_ivectors,
    max_candidates=400,
    centers=en_centers,
)
en_idf = json.loads((en_out / 'models/title_idf.json').read_text())
en_old_columns = json.loads((en_out / 'models/fusion_features.json').read_text())[
    'features'
]
en_rank_columns = [col for col in en_old_columns if col.startswith('rank_')]
en_extended = extra_features(
    en_frame, en_bq, en_bitems, en_idf, rank_columns=en_rank_columns
)
en_old = CatBoostClassifier().load_model(str(en_out / 'models/fusion_final.cbm'))
en_frame['original'] = en_old.predict_proba(model_matrix(en_frame, en_old_columns))[
    :, 1
]
en_frame['extended'] = en_final.predict_proba(model_matrix(en_extended, en_columns))[
    :, 1
]
en_pop = sorted(en_bitems.item_id, key=lambda i: (-en_counts.get(i, 0), i))[:100]


In [10]:
def en_final_predictions(score):
    predictions = predictions_from_frame(
        en_frame.assign(score=score), en_bitems, score='score', k=50
    )
    for query_id in en_bq.query_id:
        top = predictions.get(query_id, [])
        predictions[query_id] = (
            top + [item for item in en_pop if item not in set(top)]
        )[:50]
    return predictions


en_original = (
    pd.read_csv(en_out / 'submissions/answer.csv', dtype=str, keep_default_na=False)
    .set_index('query_id')
    .answer.to_dict()
)
assert {
    q: ' '.join(v) for q, v in en_final_predictions(en_frame.original).items()
} == en_original
en_path = en_out / 'submissions/answer_extended.csv'
en_validation = save_submission(
    en_final_predictions(en_frame.extended), en_bq, en_bitems, en_path
)
write_json(en_validation, en_out / 'submissions/validation_extended.json')
display(en_validation)


{'rows': 2452, 'min_candidates': 50, 'max_candidates': 50, 'valid': True}

## 6. Проверка выбранного веса

Calibration выбрал 75% расширенной модели. На comparison этот вариант дал 0.79159
против 0.79569 исходной модели. Не готовлю его к отправке: локальный выбор веса не подтвердился
даже на второй половине dev. Stacking дал 0.79661 -- слишком маленький прирост для дополнительной
метамодели с учетом повторного использования dev. Сохраняю эксперимент, финальную схему не усложняю.

In [2]:
en_cat_choices = en_blends[
    en_blends.part.eq('calibration') & en_blends.variant.str.startswith('extended_300_')
]
en_cat_choice = (
    en_cat_choices.sort_values(['objective', 'variant'], ascending=[False, True])
    .iloc[0]
    .variant
)
en_alpha = float(en_cat_choice.rsplit('_', 1)[1])
en_comparison = en_blends[en_blends.part.eq('comparison')].set_index('variant')
en_delta = float(
    en_comparison.loc[en_cat_choice, 'objective']
    - en_comparison.loc['extended_300_0', 'objective']
)
write_json(
    {
        'extended_alpha': en_alpha,
        'chosen_on': 'dev_calibration',
        'comparison_delta': en_delta,
        'recommended': False,
        'reason': 'Выбранный вес ухудшил comparison; stacking усложняет решение без убедительного выигрыша',
        'seed': en_seed,
        'baseline_exactly_reproduced': True,
        'official_score': None,
    },
    en_out / 'models/extended_blend.json',
)
display(en_blends[en_blends.variant.eq(en_cat_choice)])


,variant,part,cold,warm,objective
8,extended_300_0.75,calibration,0.811667,0.828857,0.818110
9,extended_300_0.75,comparison,0.785944,0.801000,0.791587
